# Final report · Revision after Hito 2 feedback

**NextLap · IIT414W · Team Lab 8 (G08).** This notebook answers the priority correction in the Hito 2 feedback (20 Nov) and a Demo Day question (R5):

> *With three clean laps, the proposed signed deviation equals 1.5 × (current_lap_s − recent_mean_s). Both terms are already in OLS. Also, the bootstrap interval compares T1 with the matched control. It does not measure T1 versus T3.*
> *Check: specify what information the next variant adds to those two inputs. Compare on the same IDs, by race and history length, and keep the exploratory T3 separate from the planned result. Use a direct paired comparison if assessing T1 versus T3.*

It refits the planned versions exactly as in `Hito2/hito2_experiment.ipynb` (same helper, same rows, same seed) and adds:

- **R1** the algebra check of the signed deviation, and an executed OLS with it (T4);
- **R2** what the spread adds beyond `current_lap_s` and `recent_mean_s` (decomposition, T5 exploratory);
- **R3** planned (a)/(b)/(c) by race **and** history length on the same IDs;
- **R4** a direct paired comparison T1 vs T3 (exploratory, kept separate).
- **R5** the Demo Day question: why driver 23's lap 7 was slow (field check, reconstructed lapping) and a diagnostic slice of rows where the driver was being lapped.

The nominated candidate is **not** changed: it stays the planned (c) = T1, frozen at Hito 2.
Run: `jupyter nbconvert --to notebook --execute --inplace Final/final_revision.ipynb` (or Run All). `RANDOM_SEED = 414`. No test data.

## 0. Setup (same data, helper and checks as Hito 2)

In [1]:
import hashlib, json, os, sys, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd, sklearn

HERE = (Path.cwd() if (Path.cwd() / "final_revision.ipynb").is_file() else Path.cwd() / "Final").resolve()
REPO = HERE.parent
DATA_DIR = Path(os.environ.get("NEXTLAP_DATA", REPO / "W09_StudentPack_v2" / "base_NextLap")).resolve()
OUT = HERE / "outputs"; OUT.mkdir(exist_ok=True)
RANDOM_SEED, TOL = 414, 1e-9
assert (DATA_DIR / "train_clean_v1.csv").is_file(), f"Course data not found at {DATA_DIR}. See Final/README.md."
sys.path.insert(0, str(DATA_DIR))
import nextlap_reference_v1 as R   # course helper, unmodified
np.random.seed(RANDOM_SEED)
KEYS, STINT_KEYS = R.KEYS, R.KEYS + ["stint"]
mae = lambda y, p: float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))

raw, manifest = R.load_snapshots(DATA_DIR)          # verifies snapshot checksums
train = pd.read_csv(DATA_DIR / "train_clean_v1.csv", dtype={"driver_number": str})
val = pd.read_csv(DATA_DIR / "validation_clean_v1.csv", dtype={"driver_number": str})
assert len(train) == 5772 and len(val) == 1808 and val.pair_id.is_unique and train.pair_id.is_unique
assert sorted(train["round"].unique()) == list(range(1, 8)) and sorted(val["round"].unique()) == [8, 9]
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__, "| python", platform.python_version())

numpy 2.2.6 | pandas 2.3.3 | sklearn 1.7.2 | python 3.10.12


## 1. Window features on the original timeline

Same function as Hito 2 (`history_features`), extended with the quantities needed for R1–R2. All of them use only clean laps ≤ *t* in the same driver, race and stint:

- `prev_mean_s`: mean of the **previous** clean laps in the window (excluding *t*);
- `signed_dev_s` = `current_lap_s` − `prev_mean_s` (the variant proposed at Hito 2);
- `abs_dev_s` = |`current_lap_s` − `recent_mean_s`| (magnitude of lap *t*'s deviation from the window mean);
- `prev_std_s`: spread of the window **without** lap *t* (disagreement among the earlier laps).

In [2]:
def history_features(laps, window=3, min_laps=2):
    d = laps.sort_values(KEYS + ["lap_number"]).reset_index(drop=True).copy()
    d["current_clean"] = (d.lap_number.gt(1) & d.lap_time_s.gt(0) & d.is_accurate & ~d.pit_in
                          & ~d.pit_out & ~d.deleted & d.track_status.eq("1"))
    d["hv"] = d.lap_time_s.where(d.current_clean)
    g = d.groupby(STINT_KEYS, dropna=False, sort=False)["hv"]
    d["mean_check_s"] = g.transform(lambda s: s.rolling(window, min_periods=1).mean())
    d["count_check"] = g.transform(lambda s: s.rolling(window, min_periods=1).count())
    d["recent_std_s"] = g.transform(lambda s: s.rolling(window, min_periods=min_laps).std(ddof=1))
    # window without lap t: the window ending at t, minus t itself (t is clean for every evaluated pair)
    d["prev_sum"] = g.transform(lambda s: s.rolling(window, min_periods=1).sum()) - d.hv
    d["prev_n"] = d["count_check"] - d.hv.notna()
    d["prev_mean_s"] = d.prev_sum / d.prev_n.where(d.prev_n > 0)
    d["prev_std_s"] = g.transform(lambda s: s.shift(1).rolling(window - 1, min_periods=2).std(ddof=1))
    d["has_history2"] = (d["count_check"] >= min_laps).astype(int)
    d["pair_id"] = (d.season.astype(str) + "-R" + d["round"].astype(str) + "-" + d.driver_number
                    + "-L" + d.lap_number.astype(int).astype(str))
    return d[["pair_id", "lap_time_s", "mean_check_s", "count_check", "recent_std_s", "has_history2",
              "prev_mean_s", "prev_n", "prev_std_s"]]

hist = history_features(raw)
def attach(frame):
    out = frame.merge(hist, on="pair_id", how="left", validate="one_to_one", indicator=True)
    assert out["_merge"].eq("both").all() and out.pair_id.tolist() == frame.pair_id.tolist()
    assert (out.lap_time_s - out.current_lap_s).abs().max() < TOL          # same lap t
    assert (out.mean_check_s - out.recent_mean_s).abs().max() < TOL        # Hito 2 check 1 still holds
    assert (out.count_check == out.recent_count).all()
    out = out.drop(columns=["_merge", "lap_time_s", "mean_check_s", "count_check"])
    out["signed_dev_s"] = out.current_lap_s - out.prev_mean_s              # NaN when no previous clean lap
    out["abs_dev_s"] = (out.current_lap_s - out.recent_mean_s).abs()
    return out
train, val = attach(train), attach(val)
# prev_std_s must only exist when the window has 3 laps (2 previous laps)
assert (val.prev_std_s.notna() == (val.recent_count == 3)).all()
pd.DataFrame({"split": ["train", "validation"], "rows": [len(train), len(val)],
              "recent_count_1": [int((f.recent_count == 1).sum()) for f in (train, val)],
              "recent_count_2": [int((f.recent_count == 2).sum()) for f in (train, val)],
              "recent_count_3": [int((f.recent_count == 3).sum()) for f in (train, val)]})

,split,rows,recent_count_1,recent_count_2,recent_count_3
0,train,5772,456,729,4587
1,validation,1808,148,173,1487


## R1. The signed deviation adds no new information to OLS

Window of 3 clean laps *a, b, t*: `recent_mean_s` = (a + b + t)/3, so the mean of the previous laps is (a + b)/2 = (3·mean − t)/2 and

  signed_dev = t − (a + b)/2 = **1.5 × (t − mean)** = 1.5 × (`current_lap_s` − `recent_mean_s`).

With 2 laps it is 2 × (t − mean); with 1 lap it does not exist. So on every row it is a fixed multiple of a **difference of two inputs OLS already has**. Below: the identity checked on the data, then an executed OLS with the signed deviation added (T4).

In [3]:
v = val.copy()
k = np.where(v.recent_count == 3, 1.5, np.where(v.recent_count == 2, 2.0, np.nan))
ident = (v.signed_dev_s - k * (v.current_lap_s - v.recent_mean_s)).abs()
r1_identity = {"rows_count3": int((v.recent_count == 3).sum()), "rows_count2": int((v.recent_count == 2).sum()),
               "max_abs_diff_count3_s": float(ident[v.recent_count == 3].max()),
               "max_abs_diff_count2_s": float(ident[v.recent_count == 2].max()),
               "rows_without_signed_dev": int(v.signed_dev_s.isna().sum())}
assert r1_identity["max_abs_diff_count3_s"] < 1e-9 and r1_identity["max_abs_diff_count2_s"] < 1e-9
r1_identity

{'rows_count3': 1487,
 'rows_count2': 173,
 'max_abs_diff_count3_s': 3.552713678800501e-14,
 'max_abs_diff_count2_s': 2.842170943040401e-14,
 'rows_without_signed_dev': 148}

## Fit all versions (training rows only, same pipeline as Hito 2)

Planned (pre-registered in Hito 1, reported in Hito 2): **(a)** reference, **(b)** matched control, **(c) = T1** candidate.
Exploratory, run after seeing Hito 2 validation results, **not** used to choose the nominated model: **T3** (spread, no flag), **T4** (signed deviation, Hito 2 proposal), **T5** (|t − mean| instead of the spread), **T6** (spread of the earlier laps only).

In [4]:
VERSIONS = {
    "a_reference":       (R.NUMERIC, "planned"),
    "b_control":         (R.NUMERIC + ["has_history2"], "planned"),
    "c_candidate_T1":    (R.NUMERIC + ["has_history2", "recent_std_s"], "planned"),
    "T3_no_flag":        (R.NUMERIC + ["recent_std_s"], "exploratory"),
    "T4_signed_dev":     (R.NUMERIC + ["has_history2", "signed_dev_s"], "exploratory"),
    "T5_abs_dev":        (R.NUMERIC + ["has_history2", "abs_dev_s"], "exploratory"),
    "T6_prev_spread":    (R.NUMERIC + ["has_history2", "prev_std_s"], "exploratory"),
}
models, P = {}, pd.DataFrame({"pair_id": val.pair_id, "round": val["round"], "recent_count": val.recent_count.astype(int),
                              "driver_number": val.driver_number, "y_true_s": val[R.TARGET]})
for name, (num, _) in VERSIONS.items():
    m = R.make_model("OLS", numeric=num, categorical=R.CATEGORICAL).fit(train[num + R.CATEGORICAL], train[R.TARGET])
    models[name] = m
    P[name] = m.predict(val[num + R.CATEGORICAL]); assert np.isfinite(P[name]).all()

# the planned predictions are byte-for-byte the Hito 2 ones (same IDs, same order)
h2 = pd.read_csv(REPO / "Hito2" / "outputs" / "validation_predictions.csv")
assert h2.pair_id.tolist() == P.pair_id.tolist()
same = {k: float((P[k] - h2[c]).abs().max()) for k, c in [("a_reference", "pred_a_OLS_reference"),
        ("b_control", "pred_b_matched_control"), ("c_candidate_T1", "pred_c_candidate")]}
assert max(same.values()) < TOL
P.to_csv(OUT / "final_validation_predictions.csv", index=False)
print("Planned predictions identical to Hito 2:", same)
rows = [{"version": n, "status": VERSIONS[n][1], "n": len(P), "MAE_s": mae(P.y_true_s, P[n]),
         "MAE_R8_s": mae(P.y_true_s[P["round"] == 8], P[n][P["round"] == 8]),
         "MAE_R9_s": mae(P.y_true_s[P["round"] == 9], P[n][P["round"] == 9])} for n in VERSIONS]
overall = pd.DataFrame(rows); overall["vs_control_s (b - version)"] = overall.MAE_s.iloc[1] - overall.MAE_s
overall.round(5)

Planned predictions identical to Hito 2: {'a_reference': 1.4210854715202004e-14, 'b_control': 1.4210854715202004e-14, 'c_candidate_T1': 1.4210854715202004e-14}


,version,status,n,MAE_s,MAE_R8_s,MAE_R9_s,vs_control_s (b - version)
0,a_reference,planned,1808,0.38671,0.37357,0.40558,0.00096
1,b_control,planned,1808,0.38767,0.37163,0.41071,0.00000
2,c_candidate_T1,planned,1808,0.36973,0.35879,0.38544,0.01794
3,T3_no_flag,exploratory,1808,0.36522,0.35760,0.37617,0.02245
4,T4_signed_dev,exploratory,1808,0.38775,0.37114,0.41160,-0.00008
5,T5_abs_dev,exploratory,1808,0.37240,0.36033,0.38972,0.01527
6,T6_prev_spread,exploratory,1808,0.37629,0.36173,0.39721,0.01138


**Reading R1.** T4 (signed deviation) is compared with the matched control (b): if it only rescales `current_lap_s − recent_mean_s`, OLS can already represent it on 3-lap rows, so any change in MAE can only come from 2-lap rows (factor 2 instead of 1.5) and from the filled value on 1-lap rows.

## R2. What information does the spread add beyond `current_lap_s` and `recent_mean_s`?

OLS is **linear** in its inputs: from `current_lap_s` and `recent_mean_s` it can only form *signed* linear combinations. The spread is **not** linear in the lap times. For a 3-lap window *a, b, t* (ddof = 1):

  **spread² = ¾ · (t − mean)² + ¼ · (a − b)²**

So the spread carries two things OLS cannot build from its inputs: (i) the **size** of lap *t*'s deviation regardless of sign, and (ii) the **disagreement among the earlier laps**, (a − b)², which is invisible to both the current lap and the mean. Below: the identity checked on the data, and two exploratory OLS that keep one part each (T5 = |t − mean|, T6 = spread of the earlier laps only).

In [5]:
w3 = val[val.recent_count == 3]
a_minus_b_sq = 2 * w3.prev_std_s ** 2          # two earlier laps: var(ddof=1) = (a-b)^2/2  ->  (a-b)^2 = 2*var
lhs = w3.recent_std_s ** 2
rhs = 0.75 * (w3.current_lap_s - w3.recent_mean_s) ** 2 + a_minus_b_sq / 4
r2_identity = {"rows_count3": len(w3), "max_abs_diff_s2": float((lhs - rhs).abs().max())}
assert r2_identity["max_abs_diff_s2"] < 1e-8
share = (0.75 * (w3.current_lap_s - w3.recent_mean_s) ** 2 / lhs.where(lhs > 0)).median()
r2_identity["median_share_of_spread2_from_lap_t"] = float(share)
print(r2_identity)
overall[overall.version.isin(["b_control", "c_candidate_T1", "T4_signed_dev", "T5_abs_dev", "T6_prev_spread"])].round(5)

{'rows_count3': 1487, 'max_abs_diff_s2': 2.078337502098293e-13, 'median_share_of_spread2_from_lap_t': 0.5348101265822015}


,version,status,n,MAE_s,MAE_R8_s,MAE_R9_s,vs_control_s (b - version)
1,b_control,planned,1808,0.38767,0.37163,0.41071,0.00000
2,c_candidate_T1,planned,1808,0.36973,0.35879,0.38544,0.01794
4,T4_signed_dev,exploratory,1808,0.38775,0.37114,0.41160,-0.00008
5,T5_abs_dev,exploratory,1808,0.37240,0.36033,0.38972,0.01527
6,T6_prev_spread,exploratory,1808,0.37629,0.36173,0.39721,0.01138


## R3. Planned comparison by race and history length (same IDs)

History length = `recent_count` (clean laps in the window at *t*): 1 = short history (fallback), 2, 3. Each cell uses the same `pair_id`s for every version; the last column is control − candidate (positive = candidate lower MAE).

In [6]:
def table(P, cols, by):
    out = []
    for keys, g in P.groupby(by):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(by, keys)); row["n"] = len(g)
        for c in cols: row[f"MAE_{c}"] = mae(g.y_true_s, g[c])
        out.append(row)
    return pd.DataFrame(out)
P["race"] = np.where(P["round"] == 8, "R8", "R9")
P["history"] = P.recent_count.map({1: "1 lap (fallback)", 2: "2 laps", 3: "3 laps"})
planned = ["a_reference", "b_control", "c_candidate_T1"]
r3 = pd.concat([table(P, planned, ["race", "history"]), table(P.assign(race="R8+R9"), planned, ["race", "history"]),
                table(P.assign(history="all"), planned, ["race", "history"])], ignore_index=True)
r3["control_minus_candidate_s"] = r3.MAE_b_control - r3.MAE_c_candidate_T1
r3["reference_minus_candidate_s"] = r3.MAE_a_reference - r3.MAE_c_candidate_T1
# row-weighted pooling reproduces each pooled cell
for race in ["R8", "R9", "R8+R9"]:
    sub = r3[(r3.race == race) & (r3.history != "all")]
    tot = r3[(r3.race == race) & (r3.history == "all")] if race != "R8+R9" else None
    pooled = (sub.control_minus_candidate_s * sub.n).sum() / sub.n.sum()
    ref = (P if race == "R8+R9" else P[P.race == race])
    assert abs(pooled - (mae(ref.y_true_s, ref.b_control) - mae(ref.y_true_s, ref.c_candidate_T1))) < TOL
r3.round(4)

,race,history,n,MAE_a_reference,MAE_b_control,MAE_c_candidate_T1,control_minus_candidate_s,reference_minus_candidate_s
0,R8,1 lap (fallback),82,0.3908,0.3830,0.3786,0.0045,0.0122
1,R8,2 laps,91,0.3842,0.3890,0.4131,-0.0240,-0.0289
2,R8,3 laps,893,0.3709,0.3688,0.3514,0.0174,0.0195
3,R9,1 lap (fallback),66,0.4022,0.5230,0.5464,-0.0234,-0.1442
4,R9,2 laps,82,0.4666,0.4611,0.4508,0.0102,0.0158
5,R9,3 laps,594,0.3975,0.3913,0.3585,0.0328,0.0390
6,R8+R9,1 lap (fallback),148,0.3959,0.4455,0.4534,-0.0080,-0.0575
7,R8+R9,2 laps,173,0.4233,0.4232,0.4310,-0.0078,-0.0077
8,R8+R9,3 laps,1487,0.3815,0.3778,0.3543,0.0235,0.0273
9,R8,all,1066,0.3736,0.3716,0.3588,0.0128,0.0148


## R4. Exploratory T3 vs planned T1: direct paired comparison

Per row: d = |error of T1| − |error of T3| (positive = T3 better on that row). Mean of d = MAE(T1) − MAE(T3). Interval: paired cluster bootstrap that resamples whole driver-races (seed 414, 2,000 reps), same design as the Hito 2 bootstrap. Reported overall, by race and by history length. T3 stays exploratory: it was run after seeing validation and is not the nominated model.

In [7]:
P["d_T1_minus_T3"] = (P.y_true_s - P.c_candidate_T1).abs() - (P.y_true_s - P.T3_no_flag).abs()
P["d_b_minus_T1"] = (P.y_true_s - P.b_control).abs() - (P.y_true_s - P.c_candidate_T1).abs()
def cluster_ci(frame, col, reps=2000):
    groups = frame.groupby(["round", "driver_number"]).indices; keys = list(groups)
    vals = frame[col].to_numpy(); rng = np.random.default_rng(RANDOM_SEED); boot = []
    for _ in range(reps):
        idx = np.concatenate([groups[keys[i]] for i in rng.integers(0, len(keys), len(keys))])
        boot.append(vals[idx].mean())
    boot = np.array(boot)
    return {"n": len(frame), "clusters": len(keys), "mean_s": float(vals.mean()),
            "ci95_low_s": float(np.percentile(boot, 2.5)), "ci95_high_s": float(np.percentile(boot, 97.5)),
            "share_reps_positive": float((boot > 0).mean())}
r4 = []
for label, g in [("all", P), ("R8", P[P.race == "R8"]), ("R9", P[P.race == "R9"]),
                 ("1 lap (fallback)", P[P.recent_count == 1]), ("2 laps", P[P.recent_count == 2]), ("3 laps", P[P.recent_count == 3])]:
    r4.append({"population": label, **cluster_ci(g.reset_index(drop=True), "d_T1_minus_T3")})
r4 = pd.DataFrame(r4)
# sanity: the overall mean equals the MAE difference, and the Hito 2 bootstrap (b vs T1) is reproduced
assert abs(r4.mean_s[0] - (mae(P.y_true_s, P.c_candidate_T1) - mae(P.y_true_s, P.T3_no_flag))) < TOL
h2_boot = cluster_ci(P, "d_b_minus_T1")
print("Hito 2 bootstrap reproduced (control - T1):", {k: round(v, 4) if isinstance(v, float) else v for k, v in h2_boot.items()})
r4.round(4)

Hito 2 bootstrap reproduced (control - T1): {'n': 1808, 'clusters': 43, 'mean_s': 0.0179, 'ci95_low_s': 0.0089, 'ci95_high_s': 0.0281, 'share_reps_positive': 1.0}


,population,n,clusters,mean_s,ci95_low_s,ci95_high_s,share_reps_positive
0,all,1808,43,0.0045,0.0009,0.0085,0.9910
1,R8,1066,21,0.0012,-0.0042,0.0060,0.6670
2,R9,742,22,0.0093,0.0045,0.0144,1.0000
3,1 lap (fallback),148,43,0.0371,-0.0149,0.0933,0.9135
4,2 laps,173,42,0.0047,-0.0036,0.0126,0.8555
5,3 laps,1487,42,0.0012,-0.0014,0.0038,0.8180


## R5. Demo Day question: why was lap 7 of driver 23 slow?

At the Demo Day the instructor asked whether we had investigated why driver 23 lost ~7 s on lap 7 (the lap behind the inspected case `2026-R9-23-L7`). Two checks on the released R9 timing:

1. **Field check:** did other drivers slow down on the same lap (track-wide event) or only driver 23?
2. **Traffic check:** reconstruct on-track passes. Each car's race distance at time *s* is interpolated linearly inside the lap that contains *s* (constant speed within a lap; the data have no positions or GPS). A car **laps** driver 23 during his lap *L* if its distance minus his crosses +1 between the start and end of that lap.

Then a declared **diagnostic slice** (retrospective, not a model input): validation rows where the driver was lapped by at least one car **during lap *t***, compared with the rest, control vs candidate.

In [8]:

def race_tracks(rnd):
    d = raw[raw["round"] == rnd]
    tr = {}
    for dn, g in d.groupby("driver_number"):
        g = g.sort_values("lap_number")
        tr[dn] = ((g.end_time_s - g.lap_time_s).to_numpy(), g.end_time_s.to_numpy(), g.lap_number.to_numpy(), g.driver.iloc[0])
    return tr

def distance(track, s):
    st, en, ln, _ = track
    i = np.searchsorted(en, s)
    if i >= len(en) or s < st[0]:
        return np.nan
    return ln[i] - 1 + (s - st[i]) / (en[i] - st[i])

def lapped_by(tracks, dn, lap):
    st, en, ln, _ = tracks[dn]
    k = np.flatnonzero(ln == lap)[0]
    s0, s1 = st[k] + 0.01, en[k] - 0.01
    me0, me1 = distance(tracks[dn], s0), distance(tracks[dn], s1)
    return [tracks[o][3] for o in tracks if o != dn
            and distance(tracks[o], s0) - me0 < 1 < distance(tracks[o], s1) - me1]

r9 = raw[raw["round"] == 9].sort_values(["driver_number", "lap_number"]).copy()
r9["delta_prev_s"] = r9.groupby("driver_number").lap_time_s.diff()
field = (r9[r9.lap_number.between(4, 10)].groupby("lap_number")
         .agg(drivers=("delta_prev_s", "size"), median_delta_s=("delta_prev_s", "median"),
              slower_than_prev_by_2s=("delta_prev_s", lambda x: int((x > 2).sum())),
              track_status=("track_status", lambda x: ",".join(sorted(set(x))))))
display(field)

T9 = race_tracks(9)
a23 = r9[r9.driver_number == "23"].set_index("lap_number")
case_traffic = pd.DataFrame([{"lap": int(L), "lap_time_s": a23.loc[L, "lap_time_s"],
                              "pit_in": bool(a23.loc[L, "pit_in"]), "pit_out": bool(a23.loc[L, "pit_out"]),
                              "lapped_by": ", ".join(lapped_by(T9, "23", L))} for L in range(3, 16)])
case_traffic["n_lapping_cars"] = case_traffic.lapped_by.str.split(", ").apply(lambda x: len([y for y in x if y]))
assert case_traffic.set_index("lap").loc[7, "n_lapping_cars"] == 6
display(case_traffic)


,drivers,median_delta_s,slower_than_prev_by_2s,track_status
lap_number,,,,
4.0,22,-0.1635,0,1
5.0,22,0.1570,0,1
6.0,22,-0.1165,0,1
7.0,22,0.1055,1,1
8.0,22,0.1875,0,1
9.0,22,0.2355,0,1
10.0,22,-0.0430,0,1


,lap,lap_time_s,pit_in,pit_out,lapped_by,n_lapping_cars
0,3,95.816,False,False,,0
1,4,96.460,False,False,,0
2,5,96.299,False,False,,0
3,6,98.070,False,False,LEC,1
4,7,105.128,False,False,"NOR, ANT, VER, HAM, HAD, RUS",6
5,8,97.882,False,False,,0
6,9,97.185,False,False,,0
7,10,97.124,False,False,,0
8,11,97.148,False,False,,0
9,12,100.539,False,False,"LAW, LIN",2


In [9]:

# Diagnostic slice on the 1,808 validation rows: lapped during lap t (uses other cars' timing around t; retrospective)
TR = {8: race_tracks(8), 9: race_tracks(9)}
P["lapped_during_t"] = [len(lapped_by(TR[r], dn, L)) > 0 for r, dn, L in
                        zip(val["round"], val.driver_number, val.current_lap_number)]
r5 = []
for (race, flag), g in P.groupby([P.race, P.lapped_during_t]):
    r5.append({"race": race, "lapped_during_t": flag, "n": len(g), "MAE_control": mae(g.y_true_s, g.b_control),
               "MAE_candidate": mae(g.y_true_s, g.c_candidate_T1)})
for flag, g in P.groupby("lapped_during_t"):
    r5.append({"race": "R8+R9", "lapped_during_t": flag, "n": len(g), "MAE_control": mae(g.y_true_s, g.b_control),
               "MAE_candidate": mae(g.y_true_s, g.c_candidate_T1)})
r5 = pd.DataFrame(r5); r5["control_minus_candidate_s"] = r5.MAE_control - r5.MAE_candidate
assert r5[r5.race == "R8+R9"].n.sum() == 1808
P[["pair_id", "lapped_during_t"]].to_csv(OUT / "lapped_during_t.csv", index=False)
r5.round(4)


,race,lapped_during_t,n,MAE_control,MAE_candidate,control_minus_candidate_s
0,R8,False,1018,0.3398,0.3352,0.0046
1,R8,True,48,1.0465,0.8597,0.1868
2,R9,False,707,0.3692,0.3528,0.0164
3,R9,True,35,1.2486,1.0442,0.2043
4,R8+R9,False,1725,0.3519,0.3424,0.0095
5,R8+R9,True,83,1.1317,0.9375,0.1942


## Run record

In [10]:
try:
    commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = "unknown"
run = {"command": "jupyter nbconvert --to notebook --execute --inplace Final/final_revision.ipynb",
       "git_head_at_run": commit, "seed": RANDOM_SEED, "test_used": False, "final_evaluation_data_accessed": False,
       "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
       "data_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in
                       [DATA_DIR / "train_clean_v1.csv", DATA_DIR / "validation_clean_v1.csv", DATA_DIR / "nextlap_reference_v1.py"]},
       "validation_pair_id_sha256": hashlib.sha256("\n".join(sorted(val.pair_id)).encode()).hexdigest(),
       "planned_identical_to_hito2": same, "r1_identity": r1_identity, "r2_identity": r2_identity,
       "overall": overall.to_dict("records"), "r4_T1_vs_T3": r4.to_dict("records"), "hito2_bootstrap_reproduced": h2_boot,
       "r5_case_traffic": case_traffic.to_dict("records"), "r5_slice": r5.to_dict("records"),
       "nominated_candidate": "c_candidate_T1 (unchanged since Hito 2)"}
print(json.dumps(run, indent=2, default=float))

{
  "command": "jupyter nbconvert --to notebook --execute --inplace Final/final_revision.ipynb",
  "git_head_at_run": "5e8fce011b2a9f42845edcd6837cfc51ccc39582",
  "seed": 414,
  "test_used": false,
  "final_evaluation_data_accessed": false,
  "versions": {
    "python": "3.10.12",
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "sklearn": "1.7.2"
  },
  "data_sha256": {
    "train_clean_v1.csv": "43544f38cc9ef32b44886e6751e03522601265160086fe3a4956f04b87bd5c13",
    "validation_clean_v1.csv": "b19efb6a908af12a5d2c6bdde29aa40984295e62ce0f187a201b579278db7ee8",
    "nextlap_reference_v1.py": "f6c58925cd4a477bf3e1a5a2b3968e67df3a1cffabecee9e89701c3ac2d5f994"
  },
  "validation_pair_id_sha256": "f453a98dc906904cba7307f7e25b67fbd9c9087df0a92a87477cf63422b76a57",
  "planned_identical_to_hito2": {
    "a_reference": 1.4210854715202004e-14,
    "b_control": 1.4210854715202004e-14,
    "c_candidate_T1": 1.4210854715202004e-14
  },
  "r1_identity": {
    "rows_count3": 1487,
    "rows_count2":